In [4]:
# ==========================================================
# EXERCISE 2: Grouped Analysis & Standardized Mean Difference
# Student: Viana, Mark Vincent
# ID: TUPM-26-1953
# ==========================================================

from pathlib import Path
import numpy as np
import pandas as pd

# ===== STUDENT INFO =====
STUDENT_ID = "TUPM-26-1953"
LAST_NAME = "Viana"

# Path setup
DATA_DIR = Path.cwd().parent / "data"
OUTPUT_DIR = Path.cwd().parent / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Derive parameters
id_digits = ''.join(d for d in STUDENT_ID if d.isdigit())
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
RANDOM_SEED = int(id_digits) % (2**32)
rng = np.random.default_rng(RANDOM_SEED)

# ===== LOAD & PREPARE DATA =====
measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

n_features = raw_measurements.shape[1]
feature_names = [f"Feature_{i:03d}" for i in range(n_features)]

# Identify usable features
missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)
all_missing = observed == 0
observed_min = np.where(missing, np.inf, raw_measurements).min(axis=0)
observed_max = np.where(missing, -np.inf, raw_measurements).max(axis=0)
constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))
raw_usable = raw_measurements[:, usable_indices].copy()
retained_features = [feature_names[i] for i in usable_indices]

# ===== EXERCISE 2 STARTS HERE =====
print("="*60)
print("EXERCISE 2: Grouped Analysis & Standardized Mean Difference")
print("="*60)

# 1. Build full labeled DataFrame
df = pd.DataFrame(raw_measurements.copy(), columns=feature_names)
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")
df["Is_Fail"] = df["Result"] == 1

# Keep only usable features
clean_df = df[retained_features + ["Result", "Outcome", "Is_Fail"]].copy()

# 2. Create balanced sample
pass_count = (clean_df["Outcome"] == "Pass").sum()
fail_count = (clean_df["Outcome"] == "Fail").sum()
sample_per_group = min(20 + ID_SUM, pass_count, fail_count)
print(f"Sampling {sample_per_group} records per group")

pass_sample = clean_df[clean_df["Outcome"] == "Pass"].sample(n=sample_per_group, random_state=RANDOM_SEED)
fail_sample = clean_df[clean_df["Outcome"] == "Fail"].sample(n=sample_per_group, random_state=RANDOM_SEED)
balanced_sample = pd.concat([pass_sample, fail_sample]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# 3. Select 5 features & impute
selected_pos = rng.choice(len(retained_features), size=5, replace=False)
selected_5 = [retained_features[i] for i in selected_pos]

# Impute using FULL dataset medians
full_medians = clean_df[retained_features].median()  # ← removed numeric_only=True
balanced_sample[selected_5] = balanced_sample[selected_5].fillna(full_medians)

# 4. Grouped statistics
grouped = balanced_sample.groupby("Outcome")[selected_5].agg(
    ["count", "mean", "median", "std"]  # ← simpler syntax, no rename needed
).round(4)
print("\n--- Grouped Statistics ---")
print(grouped)

# 5. Standardized Mean Difference (Cohen's d)
pass_data = balanced_sample[balanced_sample["Outcome"] == "Pass"][selected_5]
fail_data = balanced_sample[balanced_sample["Outcome"] == "Fail"][selected_5]

pass_mean = pass_data.mean()
fail_mean = fail_data.mean()
pass_var = pass_data.var(ddof=1)
fail_var = fail_data.var(ddof=1)

# Pooled within-group SD
pooled_sd = np.sqrt((pass_var + fail_var) / 2)
safe_pooled = np.where(pooled_sd > 0, pooled_sd, 1e-9)

mean_diff = fail_mean - pass_mean
std_diff = np.abs(mean_diff) / safe_pooled

# 6. Rank by Standardized Mean Difference
ex2_df = pd.DataFrame({
    "Feature": selected_5,
    "Pass_Mean": pass_mean.round(4),
    "Fail_Mean": fail_mean.round(4),
    "Absolute_Mean_Diff": np.abs(mean_diff).round(4),
    "Standardized_Mean_Diff": std_diff.round(4)
}).sort_values("Standardized_Mean_Diff", ascending=False).reset_index(drop=True)

# Save files
ex2_path1 = OUTPUT_DIR / "balanced_sample.csv"
ex2_path2 = OUTPUT_DIR / "feature_comparison.csv"
balanced_sample.to_csv(ex2_path1, index=False)
ex2_df.to_csv(ex2_path2, index=False)

# Verify NumPy ↔ Pandas match — compare directly, skip grouped lookup
numpy_sd_check = pass_data.iloc[:, 0].std(ddof=1)
pandas_sd_check = balanced_sample[balanced_sample["Outcome"]=="Pass"][selected_5[0]].std(ddof=1)
match_check = np.isclose(numpy_sd_check, pandas_sd_check)

top_feature = ex2_df.iloc[0]["Feature"]
top_smd = ex2_df.iloc[0]["Standardized_Mean_Diff"]

# ===== PRINT RESULTS =====
print("\n--- FEATURE COMPARISON (Ranked) ---")
print(ex2_df.to_string(index=False))
print(f"\nBalanced Sample Size: {len(balanced_sample)} ({sample_per_group} per group)")
print(f"Highest-Ranked Feature: {top_feature}")
print(f"Absolute Standardized Mean Difference: {top_smd:.4f}")
print(f"NumPy-Pandas Statistics Match: {match_check}")
print(f"Saved: balanced_sample.csv, feature_comparison.csv")

EXERCISE 2: Grouped Analysis & Standardized Mean Difference
Sampling 46 records per group

--- Grouped Statistics ---
        Feature_211                         Feature_118                  \
              count    mean  median     std       count    mean  median   
Outcome                                                                   
Fail             46  0.0664  0.0562  0.0459          46  0.5969  0.5974   
Pass             46  0.0596  0.0562  0.0223          46  0.5981  0.6005   

                Feature_294                               Feature_215          \
            std       count      mean    median       std       count    mean   
Outcome                                                                         
Fail     0.0076          46  608.8322  319.5496  811.0216          46  0.0837   
Pass     0.0111          46  344.3390  251.0464  319.3666          46  0.0774   

                       Feature_050                              
        median     std       count 